# 1. Imports

In [22]:
import os
import re
import json
import hashlib
from typing import List, Dict, Any, Optional, Tuple
from langchain_text_splitters import RecursiveCharacterTextSplitter
from datetime import datetime, timezone
from dataclasses import dataclass, asdict
from typing import List, Dict, Optional
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

 # 2. Law Config

In [23]:
LAW_ID = "contract_act_1872"
LAW_NAME = "Contract Act, 1872"
LAW_SHORT_NAME = "Contract Act"

PDF_PATH = "act-print-26.pdf"
COLLECTION_NAME = "bd_contract_act_1872_en_v1"
PERSIST_DIRECTORY = "chroma_contract_act_1872_v1"

EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

CHUNK_SIZE = 2000
CHUNK_OVERLAP = 300
MIN_SECTION_TEXT_LENGTH = 40
MAX_REASONABLE_SECTION = 600

# 3. PDF Load

In [24]:
documents = load_pdf_pages(PDF_PATH)

print("Loaded pages:", len(documents))
print("PDF exists:", os.path.exists(PDF_PATH))
print("First page preview:\n")
print(documents[0].page_content[:1000])

Loaded pages: 82
PDF exists: True
First page preview:

29/03/2026 The Contract Act, 1872
PRELIMINARY
The Contract Act, 1872
( ACT NO. IX OF 1872 )
1
[ 25th April, 1872 ]
  Preamble
 Whereas it is expedient to define and amend certain parts of the law relating to contracts; It is
enacted as follows:-
 
Short title 1. This Act may be called the Contract Act, 1872.
Extent
CommencementIt extends to the whole of Bangladesh; and it shall come into force on the first
day of September, 1872.
Enactments
repealed
Nothing herein contained shall affect the provisions of any Statute, Act or
Regulation not hereby expressly repealed, nor any usage or custom of trade,
nor any incident of any contract, not inconsistent with the provisions of this
Act.
Interpretation-
clause
2. In this Act the following words and expressions are used in the following
senses, unless a contrary intention appears from the context:-
    (a) When one person signifies to another his willingness to do or to abstain
from doing 

In [25]:
def sha256_file(path: str, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def load_pdf_pages(pdf_path: str):
    loader = PyPDFLoader(pdf_path)
    return loader.load()


documents = load_pdf_pages(PDF_PATH)

provenance = {
    "file_path": PDF_PATH,
    "file_name": os.path.basename(PDF_PATH),
    "sha256": sha256_file(PDF_PATH),
    "ingested_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_url": None,
    "act_name": LAW_NAME,
}

print("Loaded pages:", len(documents))
print("PDF exists:", os.path.exists(PDF_PATH))
print("First page preview:\n")
print(documents[0].page_content[:1000])

Loaded pages: 82
PDF exists: True
First page preview:

29/03/2026 The Contract Act, 1872
PRELIMINARY
The Contract Act, 1872
( ACT NO. IX OF 1872 )
1
[ 25th April, 1872 ]
  Preamble
 Whereas it is expedient to define and amend certain parts of the law relating to contracts; It is
enacted as follows:-
 
Short title 1. This Act may be called the Contract Act, 1872.
Extent
CommencementIt extends to the whole of Bangladesh; and it shall come into force on the first
day of September, 1872.
Enactments
repealed
Nothing herein contained shall affect the provisions of any Statute, Act or
Regulation not hereby expressly repealed, nor any usage or custom of trade,
nor any incident of any contract, not inconsistent with the provisions of this
Act.
Interpretation-
clause
2. In this Act the following words and expressions are used in the following
senses, unless a contrary intention appears from the context:-
    (a) When one person signifies to another his willingness to do or to abstain
from doing 

# 4. Text Cleaning

In [26]:
def normalize_legal_text(page_text: str) -> str:
    if not page_text:
        return ""

    text = page_text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove known footer/site noise
    text = re.sub(r"(?i)\bbdlaws\.minlaw\.gov\.bd/act-print-\d+\.html\b", "", text)
    text = re.sub(r"(?m)^\s*\d{1,4}/\d{1,4}\s*$", "", text)

    # Fix glued section starts like "Short title1." -> "Short title\n1. "
    text = re.sub(r"([A-Za-z\)\-])(\d{1,4}\.)", r"\1\n\2 ", text)

    # Fix glued heading/body like "CommencementIt extends..." -> "Commencement\nIt extends..."
    text = re.sub(r"([a-z\)])([A-Z])", r"\1\n\2", text)

    # Normalize spaces/newlines
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r" *\n *", "\n", text)

    return text.strip()


def build_clean_pages(documents) -> List[Dict[str, Any]]:
    clean_pages = []
    for i, doc in enumerate(documents):
        clean_pages.append({
            "page": i,
            "text": normalize_legal_text(doc.page_content),
        })
    return clean_pages


clean_pages = build_clean_pages(documents)

print("Cleaned page 0 preview:\n")
print(clean_pages[0]["text"][:2500])

Cleaned page 0 preview:

29/03/2026 The Contract Act, 1872
PRELIMINARY
The Contract Act, 1872
( ACT NO. IX OF 1872 )
1
[ 25th April, 1872 ]
Preamble
Whereas it is expedient to define and amend certain parts of the law relating to contracts; It is
enacted as follows:-

Short title 1. This Act may be called the Contract Act, 1872.
Extent
Commencement
It extends to the whole of Bangladesh; and it shall come into force on the first
day of September, 1872.
Enactments
repealed
Nothing herein contained shall affect the provisions of any Statute, Act or
Regulation not hereby expressly repealed, nor any usage or custom of trade,
nor any incident of any contract, not inconsistent with the provisions of this
Act.
Interpretation-
clause
2. In this Act the following words and expressions are used in the following
senses, unless a contrary intention appears from the context:-
(a) When one person signifies to another his willingness to do or to abstain
from doing anything, with a view to obtaining th

In [27]:
def inspect_page(page_num: int, chars: int = 2500):
    print("=" * 120)
    print(f"PAGE: {page_num}")
    print("=" * 120)
    print(clean_pages[page_num]["text"][:chars])


for p in [0, 1, 2, 3, 4, 5, 10, 20, 40, 60]:
    inspect_page(p, chars=2500)

PAGE: 0
29/03/2026 The Contract Act, 1872
PRELIMINARY
The Contract Act, 1872
( ACT NO. IX OF 1872 )
1
[ 25th April, 1872 ]
Preamble
Whereas it is expedient to define and amend certain parts of the law relating to contracts; It is
enacted as follows:-

Short title 1. This Act may be called the Contract Act, 1872.
Extent
Commencement
It extends to the whole of Bangladesh; and it shall come into force on the first
day of September, 1872.
Enactments
repealed
Nothing herein contained shall affect the provisions of any Statute, Act or
Regulation not hereby expressly repealed, nor any usage or custom of trade,
nor any incident of any contract, not inconsistent with the provisions of this
Act.
Interpretation-
clause
2. In this Act the following words and expressions are used in the following
senses, unless a contrary intention appears from the context:-
(a) When one person signifies to another his willingness to do or to abstain
from doing anything, with a view to obtaining the assent of that 

# 5. Section Parsing

In [28]:
PAGE_MARKER_TEMPLATE = "\n\n<<<PAGE:{page}>>>\n\n"


def build_full_text(clean_pages: List[Dict[str, Any]]) -> str:
    parts = []
    for item in clean_pages:
        parts.append(PAGE_MARKER_TEMPLATE.format(page=item["page"]) + item["text"])
    return "\n".join(parts)


def build_page_ranges(full_text: str) -> List[Tuple[int, int, int]]:
    matches = list(re.finditer(r"<<<PAGE:(\d+)>>>", full_text))
    ranges = []

    for idx, m in enumerate(matches):
        page_num = int(m.group(1))
        start = m.end()
        end = matches[idx + 1].start() if idx + 1 < len(matches) else len(full_text)
        ranges.append((page_num, start, end))

    return ranges


def offset_to_page(offset: int, page_ranges: List[Tuple[int, int, int]]) -> Optional[int]:
    for page_num, start, end in page_ranges:
        if start <= offset < end:
            return page_num
    return None


def parse_sections(full_text: str) -> Dict[int, Dict[str, Any]]:
    lines = full_text.splitlines()
    sections = {}

    current_section = None
    current_heading = None
    current_lines = []
    current_start_offset = None

    pending_heading_lines = []
    offset = 0

    section_start_re = re.compile(r"^\s*(\d{1,4})\.\s+(.+?)\s*$")

    def is_skippable_line(stripped: str) -> bool:
        if not stripped:
            return False
        if stripped.startswith("<<<PAGE:"):
            return False
        if "bdlaws.minlaw.gov.bd" in stripped.lower():
            return True
        if re.match(r"^\d{1,4}/\d{1,4}$", stripped):
            return True
        if stripped == LAW_NAME or stripped.endswith(LAW_NAME):
            return True
        if stripped.startswith("(") and "ACT NO." in stripped.upper():
            return True
        if re.match(r"^\[\s*\d{1,2}[a-z]{2}.+\]$", stripped, re.IGNORECASE):
            return True
        if stripped.lower() in {"preamble", "preliminary"}:
            return True
        return False

    def looks_like_heading_fragment(stripped: str) -> bool:
        if not stripped:
            return False
        if len(stripped) > 80:
            return False
        if stripped.endswith(":") or stripped.endswith(":-"):
            return False
        if re.search(r"[.;]$", stripped):
            return False
        if re.search(r"\d", stripped):
            return False
        return True

    for line in lines:
        raw_line = line
        stripped = raw_line.strip()

        line_start_offset = offset
        offset += len(raw_line) + 1

        if not stripped:
            if current_section is not None:
                current_lines.append(raw_line)
            continue

        if stripped.startswith("<<<PAGE:"):
            if current_section is not None:
                current_lines.append(raw_line)
            continue

        if is_skippable_line(stripped):
            continue

        m = section_start_re.match(stripped)

        if m:
            sec_num = int(m.group(1))
            inline_heading = m.group(2).strip()

            if 0 < sec_num <= MAX_REASONABLE_SECTION:
                # attach buffered heading fragments to this section heading
                combined_heading_parts = pending_heading_lines + [inline_heading]
                combined_heading = " ".join(combined_heading_parts).strip()
                combined_heading = re.sub(r"\s+", " ", combined_heading)

                # before switching, remove buffered heading lines from previous section body
                if current_section is not None and pending_heading_lines:
                    while current_lines and current_lines[-1].strip() in pending_heading_lines:
                        current_lines.pop()

                if current_section is not None:
                    sections[current_section] = {
                        "section_number": current_section,
                        "heading": current_heading,
                        "text": "\n".join(current_lines).strip(),
                        "start_offset": current_start_offset,
                        "end_offset": line_start_offset,
                    }

                current_section = sec_num
                current_heading = combined_heading
                current_lines = []
                current_start_offset = line_start_offset
                pending_heading_lines = []
                continue

        # buffer short heading-like lines so they can belong to next numbered section
                if looks_like_heading_fragment(stripped):
                    pending_heading_lines.append(stripped)
            continue

        pending_heading_lines = []

        if current_section is not None:
            current_lines.append(raw_line)

    if current_section is not None:
        sections[current_section] = {
            "section_number": current_section,
            "heading": current_heading,
            "text": "\n".join(current_lines).strip(),
            "start_offset": current_start_offset,
            "end_offset": len(full_text),
        }

    return sections


full_text = build_full_text(clean_pages)
page_ranges = build_page_ranges(full_text)
sections = parse_sections(full_text)

for sec, obj in sections.items():
    obj["page_start"] = offset_to_page(obj["start_offset"], page_ranges)
    obj["page_end"] = offset_to_page(max(obj["start_offset"], obj["end_offset"] - 1), page_ranges)

print("Parsed sections:", len(sections))
print("First 20 section numbers:", list(sorted(sections.keys()))[:20])

Parsed sections: 184
First 20 section numbers: [2, 3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 17, 18, 19, 20, 21, 22, 23]


In [29]:
def inspect_section_raw(sections: Dict[int, Dict[str, Any]], sec_num: int, preview_chars: int = 2000):
    if sec_num not in sections:
        print(f"Section {sec_num} NOT FOUND")
        return

    obj = sections[sec_num]
    print("=" * 120)
    print("SECTION:", obj["section_number"])
    print("HEADING:", obj["heading"])
    print("PAGES:", obj["page_start"], "-", obj["page_end"])
    print("-" * 120)
    print(obj["text"][:preview_chars])


inspect_section_raw(sections, 1, preview_chars=1500)
print("\n" + "#" * 120 + "\n")
inspect_section_raw(sections, 2, preview_chars=2000)
print("\n" + "#" * 120 + "\n")
inspect_section_raw(sections, 3, preview_chars=1200)

Section 1 NOT FOUND

########################################################################################################################

SECTION: 2
HEADING: In this Act the following words and expressions are used in the following
PAGES: 0 - 1
------------------------------------------------------------------------------------------------------------------------
senses, unless a contrary intention appears from the context:-
(a) When one person signifies to another his willingness to do or to abstain
from doing anything, with a view to obtaining the assent of that other to such
act or abstinence, he is said to make a proposal:
(b) When the person to whom the proposal is made signifies his assent
thereto, the proposal is said to be accepted. A proposal, when accepted
becomes a promise:


<<<PAGE:1>>>

CHAPTER I
OF THE COMMUNICATION, ACCEPTANCE AND REVOCATION OF PROPOSALS
(c) The person making the proposal is called the "promisor" and the person
accepting the proposal is called the 

# 6. Chunk Building

In [30]:
PAGE_TAG_RE = re.compile(r"<<<PAGE:(\d+)>>>")

def strip_page_tags(text: str) -> str:
    text = PAGE_TAG_RE.sub("", text)

    # Remove obvious running headers / dates
    text = re.sub(r"(?m)^\d{2}/\d{2}/\d{4}\s+The Contract Act,\s*1872\s*$", "", text)

    # Remove short marginal-heading lines (layout noise)
    lines = text.splitlines()
    cleaned_lines = []

    for line in lines:
        stripped = line.strip()

        if not stripped:
            cleaned_lines.append("")
            continue

        # Drop short title-like labels (Extent, Commencement, etc.)
        if len(stripped) <= 35 and re.match(r"^[A-Za-z][A-Za-z\-\s]*$", stripped):
            continue

        cleaned_lines.append(line)

    text = "\n".join(cleaned_lines)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)


def build_documents(sections: Dict[int, Dict[str, Any]], provenance: Dict[str, Any]) -> List[Document]:
    docs_for_db = []
    doc_sha = provenance["sha256"]

    for sec_num in sorted(sections.keys()):
        sec_obj = sections[sec_num]
        clean_text = strip_page_tags(sec_obj["text"])

        if len(clean_text) < MIN_SECTION_TEXT_LENGTH:
            continue

        chunks = splitter.split_text(clean_text)

        for idx, chunk_text in enumerate(chunks):
            metadata = {
                "law_id": LAW_ID,
                "law_name": LAW_NAME,
                "law_short_name": LAW_SHORT_NAME,
                "act_name": LAW_NAME,
                "section_number": sec_num,
                "section_heading": sec_obj["heading"],
                "page_start": sec_obj["page_start"],
                "page_end": sec_obj["page_end"],
                "source_pdf": provenance["file_name"],
                "source_sha256": doc_sha,
                "chunk_index": idx,
                "chunk_id": f"{doc_sha[:12]}::{LAW_ID}::{sec_num}::{idx}",
            }

            docs_for_db.append(Document(page_content=chunk_text, metadata=metadata))

    return docs_for_db


docs_for_db = build_documents(sections, provenance)

print("Total chunks prepared:", len(docs_for_db))
print("Sample metadata:\n", docs_for_db[0].metadata if docs_for_db else "No docs")
print("\nSample text:\n")
print(docs_for_db[0].page_content[:500] if docs_for_db else "No docs")

Total chunks prepared: 197
Sample metadata:
 {'law_id': 'contract_act_1872', 'law_name': 'Contract Act, 1872', 'law_short_name': 'Contract Act', 'act_name': 'Contract Act, 1872', 'section_number': 2, 'section_heading': 'In this Act the following words and expressions are used in the following', 'page_start': 0, 'page_end': 1, 'source_pdf': 'act-print-26.pdf', 'source_sha256': '93394e961e7eeddd8938bf8179ac292e7012c4dcbbca2b5161bdea0965b7299f', 'chunk_index': 0, 'chunk_id': '93394e961e7e::contract_act_1872::2::0'}

Sample text:

senses, unless a contrary intention appears from the context:-
(a) When one person signifies to another his willingness to do or to abstain
from doing anything, with a view to obtaining the assent of that other to such
act or abstinence, he is said to make a proposal:
(b) When the person to whom the proposal is made signifies his assent
thereto, the proposal is said to be accepted. A proposal, when accepted
becomes a promise:

OF THE COMMUNICATION, ACCEPTANCE AND

# 7. Sanity Check

In [31]:
def preview_chunks(docs: List[Document], n: int = 3):
    for i, doc in enumerate(docs[:n], start=1):
        md = doc.metadata
        print("=" * 100)
        print(f"Chunk #{i}")
        print("Section:", md.get("section_number"))
        print("Heading:", md.get("section_heading"))
        print("Pages:", md.get("page_start"), "-", md.get("page_end"))
        print("Text preview:")
        print(doc.page_content[:800])
        print()


preview_chunks(docs_for_db, n=3)

Chunk #1
Section: 2
Heading: In this Act the following words and expressions are used in the following
Pages: 0 - 1
Text preview:
senses, unless a contrary intention appears from the context:-
(a) When one person signifies to another his willingness to do or to abstain
from doing anything, with a view to obtaining the assent of that other to such
act or abstinence, he is said to make a proposal:
(b) When the person to whom the proposal is made signifies his assent
thereto, the proposal is said to be accepted. A proposal, when accepted
becomes a promise:

OF THE COMMUNICATION, ACCEPTANCE AND REVOCATION OF PROPOSALS
(c) The person making the proposal is called the "promisor" and the person
accepting the proposal is called the "promisee":
(d) When, at the desire of the promisor, the promisee or any other person
has done or abstained from doing, or does or abstains from doing, or
promises to do or to abstain from doing, 

Chunk #2
Section: 3
Heading: The communication of proposals, the acc

In [32]:
def inspect_sections(sections: Dict[int, Dict[str, Any]], section_numbers=None, preview_chars: int = 700):
    if section_numbers is None:
        section_numbers = list(sorted(sections.keys()))[:10]

    for sec in section_numbers:
        if sec not in sections:
            print(f"\nSection {sec} NOT FOUND")
            continue

        obj = sections[sec]
        print("\n" + "=" * 110)
        print(f"Section: {obj.get('section_number')}")
        print(f"Heading: {obj.get('heading')}")
        print(f"Pages: {obj.get('page_start')} - {obj.get('page_end')}")
        print("Preview:")
        print(obj.get("text", "")[:preview_chars])


inspect_sections(
    sections,
    section_numbers=[1, 2, 3, 4, 10]
)

print("\n" + "#" * 110 + "\n")

preview_chunks(docs_for_db, n=5)


Section 1 NOT FOUND

Section: 2
Heading: In this Act the following words and expressions are used in the following
Pages: 0 - 1
Preview:
senses, unless a contrary intention appears from the context:-
(a) When one person signifies to another his willingness to do or to abstain
from doing anything, with a view to obtaining the assent of that other to such
act or abstinence, he is said to make a proposal:
(b) When the person to whom the proposal is made signifies his assent
thereto, the proposal is said to be accepted. A proposal, when accepted
becomes a promise:


<<<PAGE:1>>>

CHAPTER I
OF THE COMMUNICATION, ACCEPTANCE AND REVOCATION OF PROPOSALS
(c) The person making the proposal is called the "promisor" and the person
accepting the proposal is called the "promisee":
(d) When, at the desire of the promisor, the promisee or an

Section: 3
Heading: The communication of proposals, the acceptance of proposals, and the
Pages: 1 - 1
Preview:
revocation of proposals and acceptances, respecti

# 8. Chroma Indexing

In [33]:
embedding = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)

vectordb = Chroma.from_documents(
    documents=docs_for_db,
    embedding=embedding,
    collection_name=COLLECTION_NAME,
    persist_directory=PERSIST_DIRECTORY,
)

vectordb.persist()

print("Chroma persisted successfully.")
print("Collection:", COLLECTION_NAME)
print("Persist directory:", PERSIST_DIRECTORY)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Chroma persisted successfully.
Collection: bd_contract_act_1872_en_v1
Persist directory: chroma_contract_act_1872_v1


# diff

In [34]:
print("=" * 100)
print("INGESTION SUMMARY")
print("=" * 100)
print("LAW_ID:", LAW_ID)
print("LAW_NAME:", LAW_NAME)
print("PDF_PATH:", PDF_PATH)
print("Loaded pages:", len(documents))
print("Parsed sections:", len(sections))
print("Prepared chunks:", len(docs_for_db))
print("Collection:", COLLECTION_NAME)
print("Persist directory:", PERSIST_DIRECTORY)

if docs_for_db:
    print("\nSample final chunk metadata:")
    print(docs_for_db[0].metadata)

print("\nNotebook ingestion pipeline completed successfully.")

INGESTION SUMMARY
LAW_ID: contract_act_1872
LAW_NAME: Contract Act, 1872
PDF_PATH: act-print-26.pdf
Loaded pages: 82
Parsed sections: 184
Prepared chunks: 197
Collection: bd_contract_act_1872_en_v1
Persist directory: chroma_contract_act_1872_v1

Sample final chunk metadata:
{'law_id': 'contract_act_1872', 'law_name': 'Contract Act, 1872', 'law_short_name': 'Contract Act', 'act_name': 'Contract Act, 1872', 'section_number': 2, 'section_heading': 'In this Act the following words and expressions are used in the following', 'page_start': 0, 'page_end': 1, 'source_pdf': 'act-print-26.pdf', 'source_sha256': '93394e961e7eeddd8938bf8179ac292e7012c4dcbbca2b5161bdea0965b7299f', 'chunk_index': 0, 'chunk_id': '93394e961e7e::contract_act_1872::2::0'}

Notebook ingestion pipeline completed successfully.
